<a href="https://colab.research.google.com/github/prabhu-shreyas/PermAware/blob/main/src/PermAware.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CODE1: install dependencies (run once in Colab)
!pip install -q transformers datasets evaluate torch scikit-learn
!pip install -q streamlit pyngrok bs4 requests

In [ ]:
# CODE2: imports, reproducibility, and file upload prompt
import os
import random
import io
import numpy as np
import pandas as pd
import torch

from google.colab import files
from sklearn.model_selection import train_test_split

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, BertForSequenceClassification

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device:", DEVICE)

# Prompt to upload dataset (Colab style)
print("Upload your CSV dataset when prompted (choose the master CSV you downloaded).")
uploaded = files.upload()  # user will select file interactively

# Read the uploaded file into a pandas DataFrame
uploaded_filename = list(uploaded.keys())[0]
print("Uploaded file:", uploaded_filename)
df = pd.read_csv(io.BytesIO(uploaded[uploaded_filename]))
print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())
print(df.head(3).to_dict(orient='records'))


In [ ]:
!ngrok config add-authtoken "Add your ngrok authentication token here"

In [ ]:
# CODE 3 — Prepare DatasetDict
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

# 1. Ensure input_text exists (it does), but keep fallback for safety
if "input_text" not in df.columns:
    df["input_text"] = df["permission"].astype(str) + " | " + df["category"].astype(str)

# 2. Use label_float for BCE
if "label_float" not in df.columns:
    raise ValueError("Dataset must contain 'label_float' column (0.0/1.0).")

df["labels"] = df["label_float"].astype(float)

# 3. Keep only useful columns
df = df[["input_text", "permission", "category", "label", "labels", "reason"]]

# 4. Stratified train/validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=RANDOM_SEED,
    stratify=df["label"]     # stratify on string Justified/Unjustified
)

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))

# 5. Convert to HF DatasetDict
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True))
})
print(dataset)

In [ ]:
# CODE 4 — Tokenization
from transformers import DataCollatorWithPadding
MODEL_NAME = "bert-base-uncased"   # You may switch to distilbert-base-uncased if preferred
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    clean_up_tokenization_spaces=True
)

def tokenize_fn(batch):
    return tokenizer(
        batch["input_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# IMPORTANT:
# Remove only input_text + metadata columns, NOT label_float (already renamed)
tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["input_text", "permission", "category", "label", "reason"]
)

# Ensure PyTorch format
tokenized.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print("Tokenization complete.")
print(tokenized)
print("Example tokenized item:", tokenized["train"][0])

In [ ]:
# CODE 5 — Corrected (robust metrics)
from transformers import Trainer, TrainingArguments, BertForSequenceClassification
import torch.nn as nn
import numpy as np
import torch

# sklearn metrics for robust evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix
)

# Load model (single logit output for BCE)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
model.to(DEVICE)

# Partial unfreeze (example: freeze first 10 layers, unfreeze last 2)
for name, param in model.named_parameters():
    if name.startswith("bert.encoder.layer.") and int(name.split('.')[3]) < 10:
        param.requires_grad = False

# Data collator (assumes tokenizer defined)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Robust compute_metrics using sklearn
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Reshape & convert
    logits = np.array(logits).reshape(-1)
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    labels = labels.astype(int).reshape(-1)

    results = {}
    # Accuracy
    results["accuracy"] = float(accuracy_score(labels, preds))

    # Precision / recall / f1 (zero_division=0 avoids exceptions)
    results["precision"] = float(precision_score(labels, preds, zero_division=0))
    results["recall"] = float(recall_score(labels, preds, zero_division=0))
    results["f1"] = float(f1_score(labels, preds, zero_division=0))

    # ROC AUC only if both classes appear in labels
    if len(np.unique(labels)) == 2:
        try:
            results["roc_auc"] = float(roc_auc_score(labels, probs))
        except Exception:
            results["roc_auc"] = float("nan")
    else:
        results["roc_auc"] = float("nan")

    # Brier score (probabilities vs binary labels)
    try:
        results["brier"] = float(brier_score_loss(labels, probs))
    except Exception:
        results["brier"] = float("nan")

    # Optional: add confusion matrix entries for debugging
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0,1]).ravel()
    results["tp"] = int(tp)
    results["tn"] = int(tn)
    results["fp"] = int(fp)
    results["fn"] = int(fn)

    return results

# TrainingArguments — tuned for your environment
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",   # more robust than roc_auc when AUC may be undefined
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# BCETrainer with **kwargs fix
class BCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels").float().to(model.device)
        # remove labels from inputs
        inputs = {k: v for k, v in inputs.items() if k != "labels"}
        outputs = model(**inputs)
        logits = outputs.logits.view(-1)
        loss_fct = nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Trainer instantiation
trainer = BCETrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train
train_result = trainer.train()
print("\nTraining finished.\n")

# Evaluate
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)

# Save model & tokenizer
trainer.save_model("./my_model_bce")
tokenizer.save_pretrained("./my_model_bce")
print("Saved model/tokenizer to ./my_model_bce")


In [ ]:
#CODE6
# 1) Save splits for reproducibility
train_df.to_csv("train_split.csv", index=False)
val_df.to_csv("val_split.csv", index=False)

# 2) Save metadata
import json
meta = {"seed": RANDOM_SEED, "model": MODEL_NAME, "num_epochs": 6, "note":"partial unfreeze, BCE single-logit"}
with open("my_model_bce/metadata.json","w") as f:
    json.dump(meta,f)
print("Saved splits and metadata.")


In [ ]:
# CODE7: inference helpers (single example + batch prediction)
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, BertForSequenceClassification
import pandas as pd

MODEL_DIR = "./my_model_bce"   # change if your model is saved elsewhere
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer & model (single-logit)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = BertForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=1)
model.to(device)
model.eval()

def predict_single(category: str, permission: str, data_safety_info: str = "", threshold: float = 0.5):
    """
    Returns dict: {'prob_justified': float, 'label': 'Justified'/'Unjustified', 'logit': float}
    """
    # Build input text (keep same format as training)
    text = f"Permission: {permission} | Category: {category}"
    # Optionally append short data_safety_info (keep small)
    if data_safety_info:
        text = text + " | Data Safety: " + data_safety_info

    # inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128).to(device)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128,
        return_token_type_ids=True
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits.view(-1)
        logit = logits.cpu().item()
        prob = float(torch.sigmoid(torch.tensor(logit)).item())
        label = "Justified" if prob >= threshold else "Unjustified"
    return {"input_text": text, "logit": logit, "prob_justified": prob, "label": label}

# Example quick test
print(predict_single("Social Media", "Audio", "App records voice messages for DMs."))

# Batch prediction helper: accepts a DataFrame with 'permission' & 'category' columns
def predict_batch(df: pd.DataFrame, data_safety_col: str = None, threshold: float = 0.5, batch_size: int = 32):
    rows = []
    texts = []
    extra = []
    for idx, r in df.iterrows():
        ds = r[data_safety_col] if data_safety_col and data_safety_col in r else ""
        text = f"Permission: {r['permission']} | Category: {r['category']}"
        if ds:
            text = text + " | Data Safety: " + str(ds)
        texts.append(text)
        extra.append((r['permission'], r['category']))
    # tokenize in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=128,
            return_token_type_ids=True
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits.view(-1).cpu().numpy()
            probs = 1.0 / (1.0 + np.exp(-logits))
        for j, p in enumerate(probs):
            perm, cat = extra[i + j]
            label = "Justified" if p >= threshold else "Unjustified"
            rows.append({"permission": perm, "category": cat, "prob_justified": float(p), "label": label})
    return pd.DataFrame(rows)

# Example usage:
# df_sample = pd.DataFrame([{"permission":"Audio","category":"Social Media"}])
# print(predict_batch(df_sample))


In [ ]:
# CODE8: quick sanity checks: show model logits/probs for a few training/validation examples
import pandas as pd
import numpy as np

# Otherwise, load a small manual test set:
test_samples = [
    {"permission":"Photos and videos", "category":"Social Media"},
    {"permission":"Photos and videos", "category":"Gaming"},
    {"permission":"Financial info", "category":"Finance"},
    {"permission":"Contacts", "category":"Productivity"},
    {"permission":"Location", "category":"Travel"}
]
test_df = pd.DataFrame(test_samples)

preds_df = predict_batch(test_df)
print(preds_df)

# Save predictions to CSV
preds_df.to_csv("sample_predictions.csv", index=False)
print("Saved sample_predictions.csv")


**STREAMLIT AND PYNGROK FOR UI**

In [ ]:
#CODE9
%%writefile app.py
# STREAMLIT CODE (AUTO-SCRAPE + MANUAL) - corrected version
import time
import requests
from bs4 import BeautifulSoup
from typing import List, Tuple
import pandas as pd
import numpy as np
import streamlit as st
import torch
from transformers import AutoTokenizer, BertForSequenceClassification

# Configuration
MODEL_DIR = "./my_model_bce"   # path where you saved the trained model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_TTL_SECONDS = 300       # 5 minutes per app_id cache

# 10 categories (approved)
CATEGORIES = [
    "Social Media", "Gaming", "Finance", "Travel", "Productivity",
    "Health & Fitness", "Education", "Communication", "E-Commerce", "Photography"
]

# canonical 14 permissions
CANONICAL_PERMS = [
    "Location","Personal info","Financial info","Health and fitness","Messages",
    "Photos and videos","Audio","Calendar","Files and docs","App activity",
    "Web browsing","App info and performance","Device or other IDs","Contacts"
]

# blacklisted phrases (these indicate noise, not a real permission)
BLACKLIST_PHRASES = [
    "encrypted", "deleted", "you can request", "you can request data",
    "you can request that", "we may collect", "we may use", "we may",
    "data is encrypted", "data is deleted", "not found", "see more",
    "show less", "learn more", "privacy policy"
]

# Load tokenizer & model
@st.cache_resource
def load_model_and_tokenizer(model_dir=MODEL_DIR):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = BertForSequenceClassification.from_pretrained(model_dir, num_labels=1)
    model.to(DEVICE)
    model.eval()
    return tokenizer, model

tokenizer, model = load_model_and_tokenizer()

# Permission normalization heuristics
def normalize_permission(raw: str) -> str:
    if not raw:
        return raw
    r = raw.lower().strip()
    # common keywords -> canonical
    mapping = {
      "microphone":"Audio","audio":"Audio","record":"Audio",
      "camera":"Photos and videos","photo":"Photos and videos","video":"Photos and videos",
      "location":"Location","gps":"Location",
      "contact":"Contacts","contacts":"Contacts",
      "financial":"Financial info","payment":"Financial info","card":"Financial info",
      "health":"Health and fitness","fitness":"Health and fitness",
      "calendar":"Calendar","events":"Calendar",
      "file":"Files and docs","document":"Files and docs","pdf":"Files and docs",
      "web":"Web browsing","browser":"Web browsing",
      "performance":"App info and performance","app info":"App info and performance",
      "device":"Device or other IDs","id":"Device or other IDs",
      "messages":"Messages","sms":"Messages","chat":"Messages",
      "personal":"Personal info","profile":"Personal info",
      "activity":"App activity"
    }
    for k, v in mapping.items():
        if k in r:
            return v
    # exact match for canonical
    for c in CANONICAL_PERMS:
        if c.lower() == r:
            return c
    # fallback: title-case (user will confirm)
    return raw.title()

# Play Store scraping + caching
@st.cache_data(ttl=CACHE_TTL_SECONDS)
def get_app_permissions_and_data_safety(app_id: str, language: str = "en") -> Tuple[List[str], str]:
    """
    Returns (raw_perms_list, data_safety_text) for the given app_id.
    Uses Play Store Data Safety page; tolerant to failures.
    """
    headers = {"User-Agent": "Mozilla/5.0 (DemoBot)"}
    base = f"https://play.google.com/store/apps/datasafety?id={app_id}&hl={language}"
    try:
        resp = requests.get(base, headers=headers, timeout=10)
        if resp.status_code != 200:
            return ([], f"Error: status {resp.status_code}")
        soup = BeautifulSoup(resp.content, "html.parser")

        # --- Extract short data safety text (if present) ---
        sec = soup.find("div", class_="UugpPb")
        data_safety_text = sec.get_text(separator=" ").strip() if sec else ""

        # --- Extract permission-like headings present on the page ---
        raw = []
        for div in soup.find_all("div", class_="Mf2Txd"):
            sec2 = div.find("div", class_="XgPdwe")
            if not sec2:
                continue
            for h3 in sec2.find_all("h3", class_="aFEzEb"):
                txt = h3.get_text(strip=True)
                if txt:
                    raw.append(txt)

        # fallback: try to find list items on the page
        if not raw:
            for li in soup.find_all("li"):
                txt = li.get_text(strip=True)
                if txt:
                    raw.append(txt)

        # filter out obviously noisy lines (blacklist)
        def is_noise(s: str) -> bool:
            if not s:
                return True
            low = s.lower()
            for phrase in BLACKLIST_PHRASES:
                if phrase in low:
                    return True
            # also ignore extremely short non-informative strings
            if len(low) < 3:
                return True
            return False

        filtered = []
        seen = set()
        for p in raw:
            if is_noise(p):
                continue
            # dedupe while preserving order
            if p not in seen:
                seen.add(p)
                filtered.append(p)

        return filtered, data_safety_text
    except Exception as e:
        return ([], f"Error scraping data safety: {e}")

# Prediction helpers (uses model loaded above)
def predict_texts(texts: List[str]) -> np.ndarray:
    """
    Input: list[str] (already formatted like training "Permission: X | Category: Y ...")
    Returns: array of probabilities (float 0..1)
    """
    all_probs = []
    batch_size = 32
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding="max_length", truncation=True, max_length=128, return_token_type_ids=True).to(DEVICE)
        with torch.no_grad():
            logits = model(**inputs).logits.view(-1).cpu().numpy()
            probs = 1.0 / (1.0 + np.exp(-logits))
        all_probs.extend(probs.tolist())
    return np.array(all_probs)

def predict_batch_from_df(df: pd.DataFrame, data_safety: str = "") -> pd.DataFrame:
    texts = []
    for _, r in df.iterrows():
        text = f"Permission: {r['permission']} | Category: {r['category']}"
        if data_safety:
            text += " | Data Safety: " + (data_safety[:200])
        texts.append(text)
    probs = predict_texts(texts)
    labels = ["Justified" if p >= 0.5 else "Unjustified" for p in probs]
    out = df.copy().reset_index(drop=True)
    out["prob_justified"] = probs.round(4)
    out["label"] = labels
    return out[["permission","category","prob_justified","label"]]

# Streamlit UI
st.set_page_config(page_title="Permission Justifier", layout="wide")
st.title("PermAware: BERT-Based App Permission Classification System")

mode = st.radio("Mode", ["Auto-scrape (recommended)", "Manual input"], index=0, horizontal=True)

# Auto-scrape UI (replaced - uses session_state to persist)
if mode == "Auto-scrape (recommended)":
    # form to accept app_id / category / include_data_safety
    with st.form("auto_form"):
        app_id = st.text_input("App ID (e.g. com.instagram.android)")
        category = st.selectbox("App Category", CATEGORIES)
        include_data_safety = st.checkbox("Use scraped Data Safety text for context (optional)", value=False)
        submitted = st.form_submit_button("Analyze")

    if submitted:
        if not app_id:
            st.warning("Please enter an App ID (package name).")
        else:
            with st.spinner("Scraping app data..."):
                raw_perms, data_safety_text = get_app_permissions_and_data_safety(app_id)
                if not raw_perms:
                    st.error(f"No permissions scraped. Details: {data_safety_text}")
                else:
                    # Normalize and dedupe
                    normalized = [normalize_permission(p) for p in raw_perms]
                    options = list(dict.fromkeys(normalized))

                    # Persist options and data_safety_text in session_state so they survive reruns
                    st.session_state["last_options"] = options
                    st.session_state["last_data_safety_text"] = data_safety_text
                    # initialize selected_perms if not present
                    if "selected_perms" not in st.session_state:
                        st.session_state["selected_perms"] = options.copy()

                    st.success(f"Found {len(options)} permission(s). Edit the list if a normalized label is incorrect, then press 'Run classification'.")

    # Show the permission editor and classification form (outside the analyze form so it's persistent)
    if "last_options" in st.session_state:
        st.write("### Permissions found (normalized). Uncheck any incorrect items before classification.")
        st.caption("Edit the list if a normalized label is incorrect, then press 'Run classification'.")

        # Use a separate form. multiselect writes into st.session_state['selected_perms'] (persistent)
        with st.form("confirm_classify_form"):
            selected = st.multiselect(
                "Permissions (edit to correct)",
                options=st.session_state["last_options"],
                default=st.session_state.get("selected_perms", st.session_state["last_options"]),
                key="selected_perms"   # binds selection to session_state['selected_perms']
            )
            classify = st.form_submit_button("Run classification on selected permissions")

        # When user clicks classify, read selection from session_state and run prediction
        if classify:
            sel = st.session_state.get("selected_perms", [])
            if not sel:
                st.info("No permissions selected for classification.")
            else:
                df_in = pd.DataFrame({"permission": sel, "category": [category]*len(sel)})
                ds_text = st.session_state.get("last_data_safety_text", "") if include_data_safety else ""
                with st.spinner("Classifying..."):
                    results_df = predict_batch_from_df(df_in, data_safety=ds_text)
                    # persist results so they render after rerun
                    st.session_state["last_results_df"] = results_df.to_dict(orient="list")
                    # optionally persist a CSV bytes for download if you want
                    st.session_state["last_results_csv"] = results_df.to_csv(index=False).encode("utf-8")

        # If results exist in session_state, render them
        if st.session_state.get("last_results_df") is not None:
            # reconstruct DataFrame
            res_df = pd.DataFrame(st.session_state["last_results_df"])
            st.write("### Results")
            st.dataframe(res_df)
            st.markdown("**Legend:** Green (high confidence justified), Amber (review), Red (likely unjustified)")
            # provide download in auto mode only
            if st.button("Download results CSV (auto-scrape)"):
                st.download_button("Download results CSV (auto-scrape)", st.session_state["last_results_csv"], "permission_results.csv")



elif mode == "Manual input":
    with st.form("manual_form"):
        category = st.selectbox("App Category", CATEGORIES, key="man_cat")
        permission = st.text_input("Permission (e.g., Audio, Location)")
        submit = st.form_submit_button("Classify permission")

    if submit:
        if not permission:
            st.warning("Please enter a permission.")
        else:
            perm_norm = normalize_permission(permission)
            df_in = pd.DataFrame([{"permission": perm_norm, "category": category}])
            with st.spinner("Classifying..."):
                results_df = predict_batch_from_df(df_in)
                st.write("### Result")
                st.dataframe(results_df)
                # Manual mode: we intentionally do NOT show a download button (per your request)
                st.markdown("_Tip: copy the table above or run bulk analysis via the Auto-scrape flow._")

# Advanced / About dataset (hidden)
with st.expander("Advanced: About dataset and diagnostics"):
    st.write("This demo uses a synthetic dataset that maps 14 canonical permissions to 10 app categories.")
    st.write("If you are debugging, you can view dataset samples and model inputs in your Colab notebook.")
    if st.button("Show last scraped raw (debug)"):
        st.write("If available, raw scraped lines (unedited) for the last request will be shown below.")
        try:
            st.write(raw_perms)
            st.write("Data safety text (short):")
            st.write(data_safety_text)
        except Exception:
            st.write("No recent scrape available in this session.")


In [ ]:
# CODE10 — Run Streamlit inside Colab using pyngrok
from pyngrok import ngrok
import subprocess

# Kill old ngrok tunnels and old streamlit processes
ngrok.kill()
!pkill -f streamlit || true

# Start streamlit server
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Expose the port 8501 to external world
public_url = ngrok.connect(addr="8501")
print("🌍 Streamlit app URL:", public_url)

**THE CODE BELOW IS FOR RESEARCH PURPOSE. NOT NECESSARY FOR BASIC IMPLEMENTATION**

**ATTENTION EXPLANATION(HTML -> PNG)**

In [ ]:
#CODE 11
# Single cell: Collated attention HTML (Light Theme A, green->amber->red heatmap)
import os, math, numpy as np, pandas as pd, torch
from IPython.display import IFrame, display, HTML

# ---------- CONFIG ----------
MODEL_DIR = "./my_model_bce"      # adjust if needed
OUT_HTML = "attn_collated_light.html"
OUT_DIR = "attn_examples"
MAX_EXAMPLES_PER_KIND = 3        # how many TP/FP/FN/TN examples to include
MAX_LEN = 128
THRESH = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUT_DIR, exist_ok=True)

# ---------- 0) Ensure model/tokenizer in memory ----------
from transformers import AutoTokenizer, BertForSequenceClassification

if 'tokenizer' not in globals() or 'model' not in globals():
    print("Reloading tokenizer/model from", MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
    model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
    model.to(DEVICE)
    model.eval()
else:
    print("Using tokenizer/model already in memory.")
    model.to(DEVICE)
    model.eval()

print("Device:", DEVICE, "Pad token id:", tokenizer.pad_token_id)

# ---------- helpers ----------
def explain_attention_pairs(text, max_len=MAX_LEN):
    """
    Returns list[(token_str, score_float)] where token_str is readable (subword cleaned)
    and score is raw cls->token attention (not normalized).
    """
    if not isinstance(text, str):
        raise ValueError("text must be a str")
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_len)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    if not hasattr(outputs, "attentions") or outputs.attentions is None:
        raise RuntimeError("Model did not return attentions. Ensure model supports output_attentions=True.")
    # stack and average over heads/layers
    attn_stack = torch.stack(list(outputs.attentions), dim=0)   # (L,B,H,S,S)
    attn_mean = attn_stack.mean(dim=2).squeeze(1).mean(dim=0)  # (S,S)
    cls_attn = attn_mean[0].cpu().numpy()                      # (S,)
    input_ids = inputs['input_ids'][0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids, skip_special_tokens=False)
    pad_id = tokenizer.pad_token_id
    pairs = []
    for tok_id, tok_str, att in zip(input_ids, tokens, cls_attn):
        if pad_id is not None and int(tok_id) == int(pad_id):
            break
        # skip CLS and SEP tokens in rendered tokens (but use their attentions in cls_attn)
        if tok_str in (tokenizer.cls_token, tokenizer.sep_token):
            continue
        # cleanup subword marker (keep readability)
        display_tok = tok_str.replace("##", "")
        pairs.append((display_tok, float(att)))
    if not pairs:
        # fallback: return the raw text as single token
        return [(text, 1.0)]
    # normalize to 0..1 per-example (so colors are relative inside each sentence)
    toks, scores = zip(*pairs)
    scores = np.array(scores, dtype=float)
    if scores.max() - scores.min() > 1e-8:
        scores = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        scores = np.zeros_like(scores)
    return list(zip(toks, scores.tolist()))

def color_from_score(score):
    """
    Map 0..1 score to a smooth gradient green -> amber -> red.
    Returns hex color string.
    """
    # green (#2ecc71) -> amber (#f1c40f) -> red (#e74c3c)
    # We'll do two-segment interpolation
    g = np.array([46, 204, 113])    # green
    a = np.array([241, 196, 15])    # amber
    r = np.array([231, 76, 60])     # red
    if score <= 0.5:
        t = score / 0.5
        rgb = (1-t)*g + t*a
    else:
        t = (score - 0.5) / 0.5
        rgb = (1-t)*a + t*r
    return '#%02x%02x%02x' % tuple([int(x) for x in rgb])

def token_span_html(tok, score):
    """
    Returns HTML span for a single token (rounded pill).
    """
    bg = color_from_score(score)
    text_color = "#111111" if score < 0.9 else "#ffffff"
    # subtle border darker than the background for crispness
    return f"<span class='token' style='background:{bg};color:{text_color};'>{tok}</span>"

# ---------- 1) Load validation/test DataFrame ----------
val_df = None
if 'val_df' in globals() and val_df is not None:
    val_df = globals()['val_df']
    print("Using existing val_df from memory:", len(val_df), "rows.")
else:
    # try common fallback files (prioritize val_split.csv if saved earlier)
    possible = ["val_split.csv", "val.csv", "validation.csv", "val_split.csv", "final_synthetic_dataset.csv"]
    found = False
    for p in possible:
        if os.path.exists(p):
            try:
                val_df = pd.read_csv(p)
                print(f"Loaded validation CSV from {p} ({len(val_df)} rows).")
                found = True
                break
            except Exception as e:
                print("Could not load", p, "-", e)
    if not found:
        raise FileNotFoundError("Validation DataFrame not in memory and no fallback CSV found. Provide `val_df` or place a CSV named val_split.csv in the working folder.")

# Ensure label column exists
label_col = None
for c in ['label_bin','labels','label','label_float','y']:
    if c in val_df.columns:
        label_col = c
        break
if label_col is None:
    raise ValueError("Could not find any label column in validation DataFrame. Columns: " + ", ".join(val_df.columns))
# Normalize labels to binary 0/1
if val_df[label_col].dtype == object:
    val_df['label_bin'] = val_df[label_col].map({'Justified':1, 'Unjustified':0})
else:
    val_df['label_bin'] = val_df[label_col].astype(float)

# Ensure input_text exists; if not, compose from permission/category
if 'input_text' not in val_df.columns:
    if 'permission' in val_df.columns and 'category' in val_df.columns:
        val_df['input_text'] = "Permission: " + val_df['permission'].astype(str) + " | Category: " + val_df['category'].astype(str)
    else:
        raise ValueError("Validation DF lacks 'input_text' and 'permission'/'category' columns. Provide input_text or those columns.")

# ---------- 2) Predict probabilities for validation set ----------
def predict_probs_batch(texts, batch_size=64):
    probs = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_LEN)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
            logits = out.logits
            if logits.shape[-1] == 1:
                logit = logits.view(-1).cpu().numpy()
                prob = 1.0 / (1.0 + np.exp(-logit))
            else:
                arr = logits.cpu().numpy()
                # probability of class 1 (assuming standard HF order)
                try:
                    prob = 1.0 / (1.0 + np.exp(-arr[:,1]))
                except Exception:
                    # fallback
                    prob = (arr[:,1] / arr.sum(axis=1)).tolist()
            probs.extend(prob.tolist())
    return np.array(probs, dtype=float)

print("Computing predictions on validation set...")
texts = val_df['input_text'].astype(str).tolist()
probs = predict_probs_batch(texts, batch_size=64)
val_df = val_df.reset_index(drop=True)
val_df['prob_justified'] = probs
val_df['pred'] = (val_df['prob_justified'] >= THRESH).astype(int)
val_df['true'] = val_df['label_bin'].astype(int)

# ---------- 3) Collect indices for TP/TN/FP/FN ----------
tp_idx = val_df[(val_df['pred']==1) & (val_df['true']==1)].index.tolist()
tn_idx = val_df[(val_df['pred']==0) & (val_df['true']==0)].index.tolist()
fp_idx = val_df[(val_df['pred']==1) & (val_df['true']==0)].index.tolist()
fn_idx = val_df[(val_df['pred']==0) & (val_df['true']==1)].index.tolist()

print(f"Validation size: {len(val_df)}  TP:{len(tp_idx)} TN:{len(tn_idx)} FP:{len(fp_idx)} FN:{len(fn_idx)}")

# ---------- 4) Save per-example HTMLs (optional) and collect items to collate ----------
def safe_make_html_for_row(idx, kind):
    r = val_df.loc[idx]
    text = str(r['input_text'])
    pairs = explain_attention_pairs(text)
    # merge small consecutive punctuation tokens into single tokens visually (optional)
    token_htmls = [token_span_html(tok, score) for tok, score in pairs]
    # Build a small card HTML for this example
    perm = r.get('permission', '')
    cat = r.get('category', '')
    prob = float(r.get('prob_justified', 0.0))
    short_input = text if len(text) <= 200 else text[:197] + "..."
    card_html = f"""
    <div class="card example">
      <div class="meta"><strong>Index:</strong> {idx} &nbsp;&nbsp; <strong>perm:</strong> {perm} &nbsp;&nbsp; <strong>cat:</strong> {cat} &nbsp;&nbsp; <strong>prob:</strong> {prob:.3f}</div>
      <div class="tokens">{" ".join(token_htmls)}</div>
      <div class="input"><em>Input:</em> {short_input}</div>
    </div>
    """
    # also save single small html file (useful for inspection)
    fname = os.path.join(OUT_DIR, f"{kind}_{idx}.html")
    with open(fname, "w", encoding="utf-8") as f:
        f.write(f"<!doctype html><meta charset='utf-8'>{card_html}")
    return card_html

# create collated sections
sections = {"True Positive": tp_idx[:MAX_EXAMPLES_PER_KIND],
            "False Positive": fp_idx[:MAX_EXAMPLES_PER_KIND],
            "False Negative": fn_idx[:MAX_EXAMPLES_PER_KIND],
            "True Negative": tn_idx[:MAX_EXAMPLES_PER_KIND]}

# ---------- 5) Build single collated HTML ----------
css = """
/* Light Theme A — IEEE-friendly */
:root {
  --bg: #ffffff;
  --card: #FAFAFA;
  --muted: #666;
  --border: #e6e6e6;
  --token-padding: 6px 10px;
  --token-radius: 10px;
}
body { background: var(--bg); font-family: "Helvetica Neue", Arial, sans-serif; color:#111; margin:28px;}
.container { max-width:1100px; margin:0 auto; }
.header { margin-bottom:18px; }
h1 { font-size:28px; margin:0 0 8px 0; }
.meta { color:var(--muted); margin-bottom:18px; }
.section-title { font-size:18px; margin:18px 0 8px 0; color:#0b6; }
.card { background: var(--card); border:1px solid var(--border); padding:14px; border-radius:8px; margin-bottom:12px; box-shadow: 0 2px 6px rgba(20,20,20,0.04); }
.card .meta { color:#333; margin-bottom:8px; font-size:13px; }
.tokens { margin-bottom:10px; display:flex; flex-wrap:wrap; gap:8px; }
.token { display:inline-block; padding: var(--token-padding); border-radius: var(--token-radius); font-size:14px; line-height:1; box-shadow: inset 0 -1px 0 rgba(0,0,0,0.06); }
.input { color:#333; font-size:13px; }
.group { margin-top:12px; margin-bottom:18px; }
.legend { margin-top:10px; font-size:13px; color:var(--muted); }
.topbar { margin-bottom: 14px; }
.badge { display:inline-block; padding:6px 10px; border-radius:8px; background:#eef7ee; color:#064; border:1px solid #dff0d8; font-weight:600; margin-right:10px;}
.small { font-size:13px; color:var(--muted); }
"""

html_parts = []
html_parts.append(f"<!doctype html><html><head><meta charset='utf-8'><title>Attention visualizations — collated</title><style>{css}</style></head><body>")
html_parts.append("<div class='container'>")
html_parts.append("<div class='header'><h1>Attention visualizations — collated</h1>")
html_parts.append(f"<div class='meta small'>Model: {MODEL_DIR} · Examples per kind: up to {MAX_EXAMPLES_PER_KIND}</div></div>")

# For each section, build set of cards
for section_name, idxs in sections.items():
    html_parts.append(f"<div class='group'><div class='section-title'>{section_name}</div>")
    if not idxs:
        html_parts.append(f"<div class='card'><div class='meta'>No examples of this kind found in validation set.</div></div>")
    else:
        for idx in idxs:
            try:
                card_html = safe_make_html_for_row(idx, section_name.replace(" ","_"))
                html_parts.append(card_html)
            except Exception as e:
                html_parts.append(f"<div class='card'><div class='meta'>Error creating example {idx}: {str(e)}</div></div>")
    html_parts.append("</div>")

html_parts.append("<div class='legend small'>Legend: token background shows attention (green→amber→red). Probabilities shown per example in the meta row.</div>")
html_parts.append("</div></body></html>")

full_html = "\n".join(html_parts)
with open(OUT_HTML, "w", encoding="utf-8") as f:
    f.write(full_html)

print("Saved collated HTML to", OUT_HTML)
# display inline (Colab may not render full file; show link and try inline IFrame)
display(HTML(f"<a href='files/{OUT_HTML}' target='_blank'>Open collated attention HTML (new tab)</a>"))
try:
    display(IFrame(OUT_HTML, width=1000, height=650))
except Exception:
    pass

# Also list saved per-example files
print("Per-example HTMLs saved to folder:", OUT_DIR)


In [ ]:
#Run only when async error occurs
# Install system dependencies Playwright needs
!apt-get update -y
!apt-get install -y \
  libatk1.0-0 libatk-bridge2.0-0 libatspi2.0-0 libxcomposite1 \
  libxrandr2 libgtk-3-0 libpangocairo-1.0-0 libcups2 libxss1 libnss3 \
  libasound2 fonts-liberation libappindicator3-1 libxshmfence1


In [ ]:
#Run only once per session
# Cell B — pip install Playwright + helper
!pip install -q playwright nest_asyncio
# install chromium for Playwright
!playwright install chromium

In [ ]:
#CODE 12
# Cell C — async Playwright renderer (requires nest_asyncio + playwright installed)
import os, asyncio, time
import nest_asyncio
nest_asyncio.apply()   # allow nested event loop inside Colab

from playwright.async_api import async_playwright
from IPython.display import Image, display

HTML_FILE = "attn_collated_light.html"
OUT_FULL = "attn_collated_light.png"
OUT_CROP = "attn_example_crop.png"

if not os.path.exists(HTML_FILE):
    raise FileNotFoundError(f"{HTML_FILE} not found. Generate collated HTML first.")

async def render(html_path, out_full=OUT_FULL, out_crop=OUT_CROP, width=1200, scale=2, selector=None, padding=8):
    url = "file://" + os.path.abspath(html_path)
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-setuid-sandbox"])
        page = await browser.new_page(viewport={"width": width, "height": 1000, "deviceScaleFactor": scale})
        await page.goto(url, wait_until="networkidle")
        await asyncio.sleep(0.6)
        await page.screenshot(path=out_full, full_page=True)
        print("Saved full page:", out_full)
        if selector:
            el = await page.query_selector(selector)
            if el is None:
                print("Selector not found:", selector)
            else:
                box = await el.bounding_box()
                if box is None:
                    print("Element bounding box is None (maybe off-screen).")
                else:
                    x = max(0, box['x'] - padding)
                    y = max(0, box['y'] - padding)
                    w = box['width'] + 2*padding
                    h = box['height'] + 2*padding
                    await page.screenshot(path=out_crop, clip={"x": x, "y": y, "width": w, "height": h})
                    print("Saved cropped element:", out_crop)
        await browser.close()
    return out_full, out_crop

# Run it
selector_example = None   # or set a CSS selector string if you want a crop
out_full, out_crop = asyncio.get_event_loop().run_until_complete(render(HTML_FILE, selector=selector_example))

# Display results
if os.path.exists(out_full):
    display(Image(out_full))
if selector_example and os.path.exists(out_crop):
    display(Image(out_crop))


**BASELINE COMPARISON**

In [ ]:
# Load train & validation splits (created earlier by CODE6)
import pandas as pd

train_df = pd.read_csv("train_split.csv")
val_df   = pd.read_csv("val_split.csv")

print("Train:", train_df.shape, "Val:", val_df.shape)


In [ ]:
#   BASELINE COMPARISON (TF-IDF + ML MODELS)
print("=== BASELINE COMPARISON STARTED ===")

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Load splits
train_df = pd.read_csv("train_split.csv")
val_df   = pd.read_csv("val_split.csv")

# Make sure text + label exist
train_df["text"] = train_df["input_text"]
val_df["text"]   = val_df["input_text"]

y_train = train_df["labels"].astype(int)
y_val   = val_df["labels"].astype(int)

# 2. TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=5000,
    min_df=2
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val   = vectorizer.transform(val_df["text"])

# 3. Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Linear SVM": LinearSVC(),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=200)
}

results = []

# 4. Train & evaluate models
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    # probability support only for some models
    try:
        prob = model.decision_function(X_val)
        prob = 1 / (1 + np.exp(-prob))
    except:
        try:
            prob = model.predict_proba(X_val)[:,1]
        except:
            prob = pred  # fallback (not ideal, but safe)

    acc = accuracy_score(y_val, pred)
    prec = precision_score(y_val, pred, zero_division=0)
    rec = recall_score(y_val, pred, zero_division=0)
    f1 = f1_score(y_val, pred, zero_division=0)

    # ROC AUC
    if len(np.unique(y_val)) == 2:
        try:
            auc = roc_auc_score(y_val, prob)
        except:
            auc = np.nan
    else:
        auc = np.nan

    results.append([name, acc, prec, rec, f1, auc])
    print(f"{name} done.")

# 5. Display results as table
baseline_results = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
)

print("\n=== BASELINE RESULTS ===")
display(baseline_results)

# Save for paper
baseline_results.to_csv("baseline_results.csv", index=False)
print("Saved baseline_results.csv")


**ABLATION STUDY**

In [ ]:
# === Ablation Study (fixed) ===
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from transformers import AutoTokenizer, BertForSequenceClassification

print("=== ABLATION STUDY STARTED ===")

# Load validation split
val_df = pd.read_csv("val_split.csv")
print("Loaded validation rows:", len(val_df))

# Load your model + tokenizer
MODEL_DIR = "./my_model_bce"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = BertForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=1)
model.to(device)
model.eval()

# Function to run prediction on custom text batch (expects list[str])
def predict(texts, batch_size=32):
    # ensure plain Python list[str]
    if isinstance(texts, (pd.Series, np.ndarray)):
        texts = texts.astype(str).tolist()
    elif not isinstance(texts, list):
        # fallback: try to coerce
        texts = list(map(str, list(texts)))
    probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=128,
            return_token_type_ids=True
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits.view(-1).cpu().numpy()
            batch_probs = 1.0 / (1.0 + np.exp(-logits))
            probs.extend(batch_probs.tolist())

    return np.array(probs, dtype=float)


def safe_roc_auc(y_true, probs):
    try:
        # roc_auc_score requires both classes present
        if len(np.unique(y_true)) == 2:
            return float(roc_auc_score(y_true, probs))
        else:
            return float("nan")
    except Exception:
        return float("nan")


def compute_metrics(y_true, probs):
    preds = (probs >= 0.5).astype(int)
    metrics = {
        "Accuracy": float(accuracy_score(y_true, preds)),
        "Precision": float(precision_score(y_true, preds, zero_division=0)),
        "Recall": float(recall_score(y_true, preds, zero_division=0)),
        "F1": float(f1_score(y_true, preds, zero_division=0)),
        "ROC-AUC": safe_roc_auc(y_true, probs),
    }
    return metrics

# Prepare y_true
y_true = val_df["labels"].astype(float).values
y_true = y_true.astype(int)

results = {}

# 1) FULL INPUT (baseline reference)
texts_full = ("Permission: " + val_df['permission'].astype(str) + " | Category: " + val_df['category'].astype(str)).astype(str).tolist()
probs_full = predict(texts_full)
results["Full Input"] = compute_metrics(y_true, probs_full)

# 2) Remove Category
texts_no_cat = ("Permission: " + val_df['permission'].astype(str)).astype(str).tolist()
probs_no_cat = predict(texts_no_cat)
results["No Category"] = compute_metrics(y_true, probs_no_cat)

# 3) Remove Permission
texts_no_perm = ("Category: " + val_df['category'].astype(str)).astype(str).tolist()
probs_no_perm = predict(texts_no_perm)
results["No Permission"] = compute_metrics(y_true, probs_no_perm)

# 4) Remove Both (empty input)
texts_empty = [""] * len(val_df)
probs_empty = predict(texts_empty)
results["No Permission + No Category"] = compute_metrics(y_true, probs_empty)

# 5) Add Noise Input
texts_noise = ("Permission: " + val_df['permission'].astype(str) + " | Category: " + val_df['category'].astype(str) + " | Noise: random unrelated text about privacy").astype(str).tolist()
probs_noise = predict(texts_noise)
results["With Noise"] = compute_metrics(y_true, probs_noise)

# Convert to DataFrame (round numbers for readability)
ablation_df = pd.DataFrame(results).T.round(4)
print("\n=== ABLATION RESULTS ===")
display(ablation_df)

# Save
ablation_df.to_csv("ablation_results.csv")
print("Saved ablation_results.csv")
